In [6]:
# %%
# Cell 1: Baseline CNN Experiment
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define Baseline model
def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, 1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU()
    )

class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(1, 32),
            conv_block(32, 64),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(9216, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

# Data loader setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_loader = DataLoader(datasets.MNIST('.', train=True, download=True, transform=transform),
                          batch_size=128, shuffle=True)
test_loader = DataLoader(datasets.MNIST('.', train=False, transform=transform),
                         batch_size=128, shuffle=False)

# Instantiate, train, and evaluate
model = BaselineCNN().to(device)
optimizer = optim.SGD(model.parameters(), lr=1e-2, momentum=0.9, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
epochs = 15

t0 = time.time()
for ep in range(1, epochs+1):
    model.train()
    running_loss = 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * data.size(0)
    print(f"Epoch {ep}/{epochs} - Train Loss: {running_loss/len(train_loader.dataset):.4f}")
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item() * data.size(0)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(test_loader.dataset)
    accuracy = correct / len(test_loader.dataset)
    print(f"          Test Loss: {test_loss:.4f} | Accuracy: {accuracy*100:.2f}")
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item() * data.size(0)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(test_loader.dataset)
    accuracy = correct / len(test_loader.dataset)
    print(f"          Test Loss: {test_loss:.4f} | Accuracy: {accuracy*100:.2f}")
t_total = time.time() - t0
print(f"Baseline total training time: {t_total:.2f}s")
total_params = sum(p.numel() for p in model.parameters())
print(f"Baseline total parameters: {total_params}")

Epoch 1/15 - Train Loss: 0.1382
          Test Loss: 0.0673 | Accuracy: 98.13
          Test Loss: 0.0673 | Accuracy: 98.13
Epoch 2/15 - Train Loss: 0.0403
          Test Loss: 0.0441 | Accuracy: 98.60
          Test Loss: 0.0441 | Accuracy: 98.60
Epoch 3/15 - Train Loss: 0.0263
          Test Loss: 0.0324 | Accuracy: 98.98
          Test Loss: 0.0324 | Accuracy: 98.98
Epoch 4/15 - Train Loss: 0.0172
          Test Loss: 0.0316 | Accuracy: 98.94
          Test Loss: 0.0316 | Accuracy: 98.94
Epoch 5/15 - Train Loss: 0.0113
          Test Loss: 0.0303 | Accuracy: 98.93
          Test Loss: 0.0303 | Accuracy: 98.93
Epoch 6/15 - Train Loss: 0.0084
          Test Loss: 0.0297 | Accuracy: 99.01
          Test Loss: 0.0297 | Accuracy: 99.01
Epoch 7/15 - Train Loss: 0.0052
          Test Loss: 0.0295 | Accuracy: 98.99
          Test Loss: 0.0295 | Accuracy: 98.99
Epoch 8/15 - Train Loss: 0.0038
          Test Loss: 0.0281 | Accuracy: 99.05
          Test Loss: 0.0281 | Accuracy: 99.05
Epoch 9/

In [7]:
# Cell 2: SVD-Fixed CNN Experiment (fixed rank = 16)
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np

# Utility: low-rank SVD Linear
class SVDLinear(nn.Module):
    def __init__(self, in_features, out_features, rank=16):
        super().__init__()
        W = torch.empty(out_features, in_features)
        nn.init.kaiming_uniform_(W, a=np.sqrt(5))
        U, S, Vh = torch.linalg.svd(W, full_matrices=False)
        max_r = min(in_features, out_features)
        self.rank = min(rank, max_r)
        self.U = nn.Parameter(U[:, :self.rank].clone())
        self.S = nn.Parameter(S[:self.rank].clone())
        self.V = nn.Parameter(Vh[:self.rank, :].clone())
        self.bias = nn.Parameter(torch.zeros(out_features))
        self.bn = nn.BatchNorm1d(out_features)

    def forward(self, x):
        h = x @ self.V.T
        h = h * self.S
        out = h @ self.U.T + self.bias
        return self.bn(out)

class SVDFixedCNN(nn.Module):
    def __init__(self, rank=16):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc1 = SVDLinear(9216, 128, rank)
        self.fc2 = SVDLinear(128, 10, rank)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# Data loader reuse
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_loader = DataLoader(datasets.MNIST('.', train=True, download=False, transform=transform),
                          batch_size=128, shuffle=True)
test_loader = DataLoader(datasets.MNIST('.', train=False, transform=transform),
                         batch_size=128, shuffle=False)

model = SVDFixedCNN(rank=16).to(device)
optimizer = optim.SGD(model.parameters(), lr=1e-2, momentum=0.9, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
epochs = 15

t0 = time.time()
for ep in range(1, epochs+1):
    model.train()
    running_loss = 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * data.size(0)
    print(f"Epoch {ep}/{epochs} - Train Loss: {running_loss/len(train_loader.dataset):.4f}")
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item() * data.size(0)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(test_loader.dataset)
    accuracy = correct / len(test_loader.dataset)
    print(f"          Test Loss: {test_loss:.4f} | Accuracy: {accuracy*100:.2f}")
t_total = time.time() - t0
print(f"SVD-Fixed total training time: {t_total:.2f}s")
total_params = sum(p.numel() for p in model.parameters())
print(f"SVD-Fixed total parameters: {total_params}")

Epoch 1/15 - Train Loss: 0.2222
          Test Loss: 0.0795 | Accuracy: 98.48
Epoch 2/15 - Train Loss: 0.0719
          Test Loss: 0.0522 | Accuracy: 98.79
Epoch 3/15 - Train Loss: 0.0486
          Test Loss: 0.0551 | Accuracy: 98.62
Epoch 4/15 - Train Loss: 0.0358
          Test Loss: 0.0403 | Accuracy: 98.95
Epoch 5/15 - Train Loss: 0.0291
          Test Loss: 0.0378 | Accuracy: 98.96
Epoch 6/15 - Train Loss: 0.0228
          Test Loss: 0.0341 | Accuracy: 99.08
Epoch 7/15 - Train Loss: 0.0186
          Test Loss: 0.0409 | Accuracy: 98.72
Epoch 8/15 - Train Loss: 0.0163
          Test Loss: 0.0340 | Accuracy: 98.97
Epoch 9/15 - Train Loss: 0.0117
          Test Loss: 0.0302 | Accuracy: 99.01
Epoch 10/15 - Train Loss: 0.0102
          Test Loss: 0.0324 | Accuracy: 99.04
Epoch 11/15 - Train Loss: 0.0081
          Test Loss: 0.0325 | Accuracy: 99.00
Epoch 12/15 - Train Loss: 0.0076
          Test Loss: 0.0342 | Accuracy: 99.04
Epoch 13/15 - Train Loss: 0.0056
          Test Loss: 0.0301 